In [1]:
import pandas as pd
import numpy as np
from collections import Counter, defaultdict

from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

# ============================================================
# CONFIG
# ============================================================
DEV_PATH  = "../data/raw/development.csv"
EVAL_PATH = "../data/raw/evaluation.csv"
SUB_PATH  = "submission_time_splittexts.csv"

MIN_RULE_SUPPORT = 30
MIN_RULE_PURITY  = 0.95
RULE_PRIORITY    = "best_purity_then_freq"
C_VALUE          = 1.5

# ============================================================
# LOAD
# ============================================================
df_dev  = pd.read_csv(DEV_PATH)
df_eval = pd.read_csv(EVAL_PATH)

# ============================================================
# TIMESTAMP PARSE + DROP BAD DEV ROWS ONLY
# ============================================================
df_dev["timestamp"]  = pd.to_datetime(df_dev["timestamp"], errors="coerce")
df_eval["timestamp"] = pd.to_datetime(df_eval["timestamp"], errors="coerce")

before = len(df_dev)
df_dev = df_dev[df_dev["timestamp"].notna()].reset_index(drop=True)
print(f"Dropped DEV rows with bad timestamp: {before - len(df_dev)}")

print("DEV samples:", len(df_dev))
print("EVAL samples:", len(df_eval))

# ============================================================
# BASIC FIXES
# ============================================================
for df in (df_dev, df_eval):
	df["article"] = df["article"].fillna("").astype(str)
	df["title"]   = df["title"].fillna("").astype(str)
	df["source"]  = df["source"].fillna("").astype(str)

	# keep them separate (lowercase)
	df["title_text"]   = df["title"].str.lower()
	df["article_text"] = df["article"].str.lower()

# ============================================================
# NUMERIC FEATURES (TIME ON)
# ============================================================
def add_numeric(df):
	df["n_tokens"]    = df["article"].str.split().str.len()
	df["title_len"]   = df["title"].str.len()
	df["article_len"] = df["article"].str.len()
	df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

	df["year"]  = df["timestamp"].dt.year
	df["month"] = df["timestamp"].dt.month
	df["dow"]   = df["timestamp"].dt.dayofweek
	return df

df_dev  = add_numeric(df_dev)
df_eval = add_numeric(df_eval)

NUM_COLS = [
	"n_tokens", "title_len", "article_len", "title_ratio",
	"year", "month", "dow"
]

# Fill numeric safely:
# - DEV has valid timestamp by construction => year/month/dow should be ok
# - EVAL may have NaT => encode missing time as -1 (not 0)
for df in (df_dev, df_eval):
	df[["year", "month", "dow"]] = df[["year", "month", "dow"]].fillna(-1)
	df[["n_tokens", "title_len", "article_len", "title_ratio"]] = \
		df[["n_tokens", "title_len", "article_len", "title_ratio"]].replace([np.inf, -np.inf], 0).fillna(0)

# ============================================================
# FEATURES (SEPARATED TEXTS)
# ============================================================
FEATURES = ["source", "title_text", "article_text"] + NUM_COLS

X_dev  = df_dev[FEATURES]
y_dev  = df_dev["label"].astype(int).values
X_eval = df_eval[FEATURES]

# ============================================================
# RULES (recommend: article-only)
# ============================================================
def tokenize_for_rules(text: str):
	return text.split()

def mine_pure_rules(texts, labels):
	counts = defaultdict(lambda: Counter())
	for txt, y in zip(texts, labels):
		for tok in set(tokenize_for_rules(txt)):
			counts[tok][int(y)] += 1

	rule_token_to_class = {}
	rule_meta = {}
	for tok, c in counts.items():
		total = sum(c.values())
		if total < MIN_RULE_SUPPORT:
			continue
		best_class, best_freq = c.most_common(1)[0]
		purity = best_freq / total
		if purity >= MIN_RULE_PURITY:
			rule_token_to_class[tok] = int(best_class)
			rule_meta[tok] = (float(purity), int(total))
	return rule_token_to_class, rule_meta

def apply_rules(texts, rule_token_to_class, rule_meta):
	rule_pred = np.full(len(texts), -1, dtype=int)
	for i, txt in enumerate(texts):
		toks = set(tokenize_for_rules(txt))
		hits = [t for t in toks if t in rule_token_to_class]
		if not hits:
			continue
		# best purity then support
		hits.sort(key=lambda t: (rule_meta[t][0], rule_meta[t][1]), reverse=True)
		rule_pred[i] = int(rule_token_to_class[hits[0]])
	return rule_pred

# ============================================================
# MODEL (TF-IDF SEPARATI)
# ============================================================
def make_model():
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),

			# TITLE TF-IDF
			("w_title", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, 2),
				min_df=2,
				max_df=0.95,
				sublinear_tf=True,
				max_features=80_000
			), "title_text"),

			("c_title", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, 5),
				min_df=2,
				max_df=0.95,
				sublinear_tf=True,
				max_features=120_000
			), "title_text"),

			# ARTICLE TF-IDF
			("w_art", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, 2),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=250_000
			), "article_text"),

			("c_art", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, 5),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=300_000
			), "article_text"),

			("num", StandardScaler(), NUM_COLS)
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C_VALUE,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([("pre", pre), ("clf", clf)])

# ============================================================
# TRAIN
# ============================================================
print("\nTraining model on full development...")
model = make_model()
model.fit(X_dev, y_dev)

print("Mining rules on full development (article-only)...")
rule_token_to_class, rule_meta = mine_pure_rules(df_dev["article_text"], y_dev)
print("Total rules:", len(rule_token_to_class))

# ============================================================
# PREDICT
# ============================================================
print("Predicting on evaluation...")
model_pred = model.predict(X_eval)

rule_pred = apply_rules(df_eval["article_text"], rule_token_to_class, rule_meta)

final_pred = model_pred.copy()
mask = (rule_pred != -1)
final_pred[mask] = rule_pred[mask]

print(f"Rule coverage on eval: {mask.mean():.3f}")

# ============================================================
# SUBMISSION (BLINDATA)
# ============================================================
final_pred = np.asarray(final_pred, dtype=int)

submission = pd.DataFrame({
	"Id": df_eval["Id"].values,
	"Predicted": final_pred
})

assert submission.isna().sum().sum() == 0
assert len(submission) == len(df_eval)

submission.to_csv(SUB_PATH, index=False)
print("Submission saved to:", SUB_PATH)



Dropped DEV rows with bad timestamp: 27750
DEV samples: 52247
EVAL samples: 20000

Training model on full development...
Mining rules on full development (article-only)...
Total rules: 83
Predicting on evaluation...
Rule coverage on eval: 0.054
Submission saved to: submission_time_splittexts.csv
